In [1]:
import pandas as pd
import os

# Define the file path (update the filename if it differs from what you downloaded)
file_path = 'dataset/Churn_Modelling.csv'

# Check if the file exists before loading
if os.path.exists(file_path):
    print("Loading Customer Churn dataset...")
    df = pd.read_csv(file_path)
    
    # Display the first few rows and dataset information
    display(df.head())
    print("\nDataset Info:")
    df.info()
    
    # Check the class distribution for the churn column (usually 'Exited' or 'Churn')
    if 'Exited' in df.columns:
        print("\nChurn Distribution (0 = Stayed, 1 = Exited):")
        print(df['Exited'].value_counts(normalize=True) * 100)
    elif 'Churn' in df.columns:
        print("\nChurn Distribution:")
        print(df['Churn'].value_counts(normalize=True) * 100)
else:
    print(f"File not found at {file_path}. Please check your dataset folder and update the filename in the code if necessary.")

Loading Customer Churn dataset...


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0



Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB

Churn Distribution (0 = Stayed, 1 = Exited):
Exited
0    79.63
1    20.37
Name: proportion, d

In [2]:
from sklearn.model_selection import train_test_split

print("Dropping irrelevant identifier columns...")
# Drop columns that don't help predict churn
df_cleaned = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

print("Encoding categorical variables (Geography and Gender)...")
# Convert categorical text data into numerical format using One-Hot Encoding
df_encoded = pd.get_dummies(df_cleaned, columns=['Geography', 'Gender'], drop_first=True)

# Define features (X) and target (y)
X = df_encoded.drop('Exited', axis=1)
y = df_encoded['Exited']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Features matrix (X) shape after encoding: {X.shape}")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
display(X_train.head())

Dropping irrelevant identifier columns...
Encoding categorical variables (Geography and Gender)...
Features matrix (X) shape after encoding: (10000, 11)
Training data shape: (8000, 11)
Testing data shape: (2000, 11)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2151,753,57,7,0.00,1,1,0,159475.08,False,False,True
8392,739,32,3,102128.27,1,1,0,63981.37,True,False,True
5006,755,37,0,113865.23,2,1,1,117396.25,True,False,False
4117,561,37,5,0.00,2,1,0,83093.25,False,False,True
7182,692,49,6,110540.43,2,0,1,107472.99,True,False,True


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Scaling numerical features...")
scaler = StandardScaler()

# Fit the scaler on the training data and transform both datasets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Random Forest model...")
# Initialize the Random Forest model
# class_weight='balanced' helps again here since only ~20% of customers churned
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Train the model
model.fit(X_train_scaled, y_train)

print("Making predictions on the test set...")
# Predict on the unseen testing data
y_pred = model.predict(X_test_scaled)

# Calculate and display the results
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed (0)', 'Exited (1)']))

Scaling numerical features...
Training Random Forest model...
Making predictions on the test set...

Test Accuracy: 85.75%

Classification Report:
              precision    recall  f1-score   support

  Stayed (0)       0.87      0.97      0.92      1593
  Exited (1)       0.77      0.43      0.55       407

    accuracy                           0.86      2000
   macro avg       0.82      0.70      0.73      2000
weighted avg       0.85      0.86      0.84      2000



In [4]:
print("Saving predictions to CSV...")

# Create a copy of the test features to store our results
test_results = X_test.copy()
test_results['Actual_Exited'] = y_test
test_results['Predicted_Exited'] = y_pred

# Save to the dataset folder
output_path = 'dataset/predicted_churn.csv'
test_results.to_csv(output_path, index=False)

print(f"Task 3 Complete! Predictions successfully saved to {output_path}")
display(test_results[['Age', 'Balance', 'Actual_Exited', 'Predicted_Exited']].head())

Saving predictions to CSV...
Task 3 Complete! Predictions successfully saved to dataset/predicted_churn.csv


,Age,Balance,Actual_Exited,Predicted_Exited
5702,36,0.00,0,0
3667,33,131023.76,0,0
1617,40,0.00,0,0
5673,34,139393.19,0,0
4272,34,77826.80,0,0
